# 01 — Data Understanding

**Phase 1** of the e-commerce churn portfolio: frame the ML problem, document the feature dictionary, profile schema / missingness / target balance, list data-quality issues, and persist a validated interim snapshot for downstream notebooks.

**Reproducible load path:** `data/raw/E_Commerce_Dataset.xlsx` → `src.data.load_and_validate` → `data/interim/ecomm_validated.*`

## 1. Business context and ML problem framing

### Business context

E-commerce platforms lose revenue when active buyers stop ordering. Acquisition cost typically exceeds retention cost, so identifying customers at risk of churn enables targeted interventions (win-back offers, service recovery, cashback, UX fixes) before the relationship ends.

### Churn definition (this dataset)

- **Unit of analysis:** one row = one customer (`CustomerID`).
- **Target:** `Churn` ∈ {0, 1}, where **1 = churned** and **0 = retained** (source "Churn Flag").
- **Prediction task:** binary classification of churn probability from demographic, engagement, fulfilment, and satisfaction features.

### Decision use-cases

| Use-case | Who acts | What the model enables |
|----------|----------|------------------------|
| Proactive retention | CRM / lifecycle marketing | Rank customers by churn risk; allocate coupons / cashback / outreach budget |
| Service recovery | CX operations | Prioritise complaint follow-up when risk is high and satisfaction is low |
| Product / category strategy | Category managers | Spot categories or devices associated with elevated churn for assortment / UX work |
| Fulfilment policy | Ops / logistics | Flag distance / recency patterns that correlate with attrition |

### Cost framing (FN vs FP) → metric choice

| Error | Business meaning | Typical cost |
|-------|------------------|--------------|
| **False negative (missed churner)** | No intervention; customer leaves | Lost LTV + re-acquisition — usually **higher** |
| **False positive (false alarm)** | Unnecessary offer / contact | Coupon cost + mild CX friction — usually **lower** |

Because churn is the minority class (~17%) and missed churners are costly, we **de-prioritise Accuracy** and favour **Recall**, **F1**, and **PR-AUC** (with Precision monitored for campaign economics). Class weights / `scale_pos_weight` will be justified in the baselines notebook.

In [8]:
from IPython.display import display
import pandas as pd

from src.data import (
    build_feature_dictionary,
    list_data_quality_issues,
    load_and_validate,
    load_data_dict,
    profile_frame,
    save_interim_snapshot,
    target_class_balance,
)
from src.utils.config import load_config
from src.utils.logging import setup_logging
from src.utils.seeding import seed_everything

setup_logging()
cfg = load_config()
seed_everything(cfg.random_seed)

print(f"Project root : {cfg.paths.root}")
print(f"Raw dataset  : {cfg.paths.raw_dataset}")
print(f"Interim dir  : {cfg.paths.interim}")
print(f"Random seed  : {cfg.random_seed}")
print(f"Target / ID  : {cfg.target_column} / {cfg.id_column}")

Project root : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction
Raw dataset  : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\raw\E_Commerce_Dataset.xlsx
Interim dir  : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\interim
Random seed  : 42
Target / ID  : Churn / CustomerID


## 2. Load and validate the observation sheet

Uses `load_and_validate` so schema checks (expected columns, unique `CustomerID`, binary `Churn`) run on every reproducible load.

In [2]:
df, validation = load_and_validate(config=cfg)

print(f"Validation OK : {validation.is_valid}")
print(f"Shape         : {df.shape[0]:,} rows × {df.shape[1]} columns")
if validation.warnings:
    print("Warnings:")
    for w in validation.warnings:
        print(f"  - {w}")

display(df.head())

2026-07-19 13:46:31 | INFO     | src.data.loader | Loading workbook from C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\raw\E_Commerce_Dataset.xlsx
2026-07-19 13:46:33 | INFO     | src.data.loader | Loaded 'E Comm': shape=(5630, 20), columns=20
2026-07-19 13:46:33 | WARNING  | src.data.validation | Columns with missing values (top): DaySinceLastOrder=5.5%, OrderAmountHikeFromlastYear=4.7%, Tenure=4.7%, OrderCount=4.6%, CouponUsed=4.5%, HourSpendOnApp=4.5%, WarehouseToHome=4.5%
2026-07-19 13:46:33 | INFO     | src.data.validation | Validation passed: rows=5630, columns=20


Validation OK : True
Shape         : 5,630 rows × 20 columns
Warnings:
  - Columns with missing values (top): DaySinceLastOrder=5.5%, OrderAmountHikeFromlastYear=4.7%, Tenure=4.7%, OrderCount=4.6%, CouponUsed=4.5%, HourSpendOnApp=4.5%, WarehouseToHome=4.5%


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,159.93
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,120.90
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120.28
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134.07
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,129.60


## 3. Feature dictionary (source + business definitions)

The Excel `Data Dict` sheet is messy (unnamed headers, typos such as "Discerption"). We parse it and enrich each variable with a business-facing definition for modelling and stakeholder communication.

In [3]:
raw_dict = load_data_dict(config=cfg)
feature_dict = build_feature_dictionary(raw_dict)

print(f"Data Dict rows parsed: {len(feature_dict)}")
display(
    feature_dict[
        ["variable", "role", "inferred_type", "source_description", "business_definition"]
    ]
)

dict_path = cfg.paths.interim / "feature_dictionary.csv"
cfg.paths.ensure_directories()
feature_dict.to_csv(dict_path, index=False)
print(f"Wrote feature dictionary → {dict_path}")

2026-07-19 13:46:33 | INFO     | src.data.loader | Loading workbook from C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\raw\E_Commerce_Dataset.xlsx
2026-07-19 13:46:33 | INFO     | src.data.loader | Loaded 'Data Dict': shape=(21, 4)


Data Dict rows parsed: 20


,variable,role,inferred_type,source_description,business_definition
0,CustomerID,id,id,Unique customer ID,Unique customer primary key. Used for joins an...
1,Churn,target,binary,Churn Flag,"Binary churn flag (1 = churned, 0 = retained)...."
2,Tenure,feature,numeric,Tenure of customer in organization,Months the customer has been with the platform...
3,PreferredLoginDevice,feature,nominal,Preferred login device of customer,Device used most often to access the storefron...
4,CityTier,feature,ordinal,City tier,Ordinal city development tier (1-3). Captures ...
5,WarehouseToHome,feature,numeric,Distance in between warehouse to home of customer,Distance from fulfilment warehouse to the cust...
6,PreferredPaymentMode,feature,nominal,Preferred payment method of customer,Preferred payment method. Note: 'CC'/'Credit C...
7,Gender,feature,nominal,Gender of customer,Self-reported gender (Female / Male).
8,HourSpendOnApp,feature,numeric,Number of hours spend on mobile application or...,Hours spent on the app/website. Engagement pro...
9,NumberOfDeviceRegistered,feature,numeric,Total number of deceives is registered on part...,Count of devices registered to the account. Mu...


Wrote feature dictionary → C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\interim\feature_dictionary.csv


## 4. Schema profile: dtypes, cardinality, missingness, duplicates

In [4]:
profile = profile_frame(
    df,
    id_column=cfg.id_column,
    target_column=cfg.target_column,
)

print(f"Rows              : {profile['n_rows']:,}")
print(f"Columns           : {profile['n_columns']}")
print(f"Duplicate IDs     : {profile['n_duplicate_ids']}")
print(f"Churn rate        : {profile['churn_rate']:.2%}")

dtype_summary = profile["dtype_summary"]
display(dtype_summary.sort_values("pct_missing", ascending=False))

cat_like = dtype_summary[dtype_summary["n_unique"] <= 20]["column"].tolist()
print("\nValue counts (cardinality ≤ 20):")
for col in cat_like:
    if col == cfg.id_column:
        continue
    print(f"\n=== {col} ===")
    display(df[col].value_counts(dropna=False))

Rows              : 5,630
Columns           : 20
Duplicate IDs     : 0
Churn rate        : 16.84%


,column,dtype,n_unique,n_missing,pct_missing
18,DaySinceLastOrder,float64,22,307,0.054529
15,OrderAmountHikeFromlastYear,float64,16,265,0.047069
2,Tenure,float64,36,264,0.046892
17,OrderCount,float64,16,258,0.045826
16,CouponUsed,float64,17,256,0.045471
8,HourSpendOnApp,float64,6,255,0.045293
5,WarehouseToHome,float64,34,251,0.044583
0,CustomerID,int64,5630,0,0.000000
12,MaritalStatus,object,3,0,0.000000
14,Complain,int64,2,0,0.000000



Value counts (cardinality ≤ 20):

=== Churn ===


Churn
0    4682
1     948
Name: count, dtype: int64


=== PreferredLoginDevice ===


...

OrderAmountHikeFromlastYear
14.0    750
13.0    741
12.0    728
15.0    542
11.0    391
16.0    333
18.0    321
19.0    311
17.0    297
NaN     265
20.0    243
21.0    190
22.0    184
23.0    144
24.0     84
25.0     73
26.0     33
Name: count, dtype: int64


=== CouponUsed ===


CouponUsed
1.0     2105
2.0     1283
0.0     1030
3.0      327
NaN      256
4.0      197
5.0      129
6.0      108
7.0       89
8.0       42
10.0      14
9.0       13
11.0      12
12.0       9
13.0       8
14.0       5
16.0       2
15.0       1
Name: count, dtype: int64


=== OrderCount ===


OrderCount
2.0     2025
1.0     1751
3.0      371
NaN      258
7.0      206
4.0      204
5.0      181
8.0      172
6.0      137
9.0       62
12.0      54
11.0      51
10.0      36
14.0      36
15.0      33
13.0      30
16.0      23
Name: count, dtype: int64

## 5. Target balance and metric implications

In [5]:
balance = target_class_balance(df, target_column=cfg.target_column)
display(balance)

churn_n = int(balance.loc[balance["class"] == 1, "count"].iloc[0])
retain_n = int(balance.loc[balance["class"] == 0, "count"].iloc[0])
churn_p = float(balance.loc[balance["class"] == 1, "proportion"].iloc[0])

print(
    f"Retained={retain_n:,} ({1 - churn_p:.1%}) | "
    f"Churned={churn_n:,} ({churn_p:.1%})"
)
print(
    "Imbalance ratio (retained:churned) ≈ "
    f"{retain_n / churn_n:.2f}:1"
)
print(
    "Primary metrics going forward: Recall, F1, PR-AUC "
    "(Accuracy is a secondary sanity check only)."
)

,class,count,proportion,label
0,0,4682,0.831616,Retained
1,1,948,0.168384,Churned


Retained=4,682 (83.2%) | Churned=948 (16.8%)
Imbalance ratio (retained:churned) ≈ 4.94:1
Primary metrics going forward: Recall, F1, PR-AUC (Accuracy is a secondary sanity check only).


## 6. Known data-quality issues

Structured findings from `list_data_quality_issues` — feeding Phase 2 EDA and Phase 3 preprocessing decisions (imputation, category harmonisation). No cleaning is applied here; this notebook only documents.

In [6]:
issues = list_data_quality_issues(
    df,
    id_column=cfg.id_column,
    target_column=cfg.target_column,
)
issues_df = pd.DataFrame(issues)
display(issues_df)

issues_path = cfg.paths.interim / "data_quality_issues.csv"
issues_df.to_csv(issues_path, index=False)
print(f"Wrote DQ issue list → {issues_path}")

,severity,category,detail
0,medium,missing_values,7 numeric feature(s) contain nulls: DaySinceLa...
1,info,duplicate_ids,CustomerID values are unique (no duplicates).
2,medium,category_overlap,PreferredLoginDevice contains overlapping labe...
3,medium,category_overlap,PreferredPaymentMode contains overlapping labe...
4,medium,category_overlap,PreferredPaymentMode contains overlapping labe...
5,medium,category_overlap,PreferedOrderCat contains overlapping labels [...
6,low,naming,Column 'PreferedOrderCat' retains source spell...
7,info,class_imbalance,Churn prevalence is 16.8%. Prefer Recall / F1 ...


Wrote DQ issue list → C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\interim\data_quality_issues.csv


### Data-quality summary (narrative)

1. **Missing values (~4.5–5.5%)** on seven numeric engagement / fulfilment fields (`DaySinceLastOrder`, `Tenure`, `OrderAmountHikeFromlastYear`, `OrderCount`, `CouponUsed`, `HourSpendOnApp`, `WarehouseToHome`). Strategy TBD in Phase 3 (likely median/mode with train-only fit).
2. **Category label overlaps:** `Phone` vs `Mobile Phone`; `CC` vs `Credit Card`; `COD` vs `Cash on Delivery`; `Mobile` vs `Mobile Phone` in order category — split signal risk until harmonised.
3. **Naming:** `PreferedOrderCat` keeps the source spelling for schema compatibility.
4. **IDs:** `CustomerID` is unique — no duplicate-key repair needed.
5. **Class imbalance:** ~16.8% churn — metric and weighting choices as framed above.

## 7. Persist validated interim snapshot

Writes the untouched-but-validated observation table plus JSON metadata (and companion dictionary / DQ CSVs above) under `data/interim/` for Phase 2+.

In [7]:
table_path, meta_path = save_interim_snapshot(
    df,
    validation=validation,
    config=cfg,
    extra_metadata={
        "phase": 1,
        "notebook": "01_Data_Understanding.ipynb",
        "churn_rate": profile["churn_rate"],
        "n_duplicate_ids": profile["n_duplicate_ids"],
        "feature_dictionary": "feature_dictionary.csv",
        "data_quality_issues": "data_quality_issues.csv",
    },
)

print(f"Interim table : {table_path}")
print(f"Metadata      : {meta_path}")
print(f"Exists        : table={table_path.exists()}, meta={meta_path.exists()}")

2026-07-19 13:46:34 | INFO     | src.data.loader | Saved interim snapshot: C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\interim\ecomm_validated.parquet (+ ecomm_validated_meta.json)


Interim table : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\interim\ecomm_validated.parquet
Metadata      : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction\data\interim\ecomm_validated_meta.json
Exists        : table=True, meta=True
